# GCN-MAPPO Adaptive Traffic Signal Control
## Manhattan 4x4 Grid — Kaggle Training
**Run cells one by one in order. Do not skip.**

In [ ]:
# CELL 1: Install SUMO (takes 2-3 min, needs Internet ON)
import subprocess, sys
print('Installing SUMO...')
subprocess.run(['apt-get', 'install', '-y', 'sumo', 'sumo-tools', 'sumo-doc'], capture_output=True)
import os
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.append('/usr/share/sumo/tools')
result = subprocess.run(['sumo', '--version'], capture_output=True, text=True)
if 'SUMO' in result.stdout:
    print('SUMO installed:', result.stdout.split('\n')[0])
else:
    print('ERROR: SUMO not found')
    print(result.stderr)

In [ ]:
# CELL 2: Copy dataset files to working directory
import os, sys, shutil, glob
WORK_DIR = '/kaggle/working'
os.chdir(WORK_DIR)

def find_dataset_files():
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'gcn_encoder.py' in files:
            print(f'Found dataset at: {root}')
            return root
    return None

dataset_dir = find_dataset_files()

if dataset_dir is None:
    print('ERROR: Could not find dataset files.')
    print('Add your dataset in the Input panel on the right.')
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        for f in files:
            print(f'{indent}  {f}')
else:
    copied = 0
    for f in glob.glob(os.path.join(dataset_dir, '*')):
        dest = os.path.join(WORK_DIR, os.path.basename(f))
        if not os.path.exists(dest):
            shutil.copy2(f, dest)
            print(f'Copied: {os.path.basename(f)}')
            copied += 1
        else:
            print(f'Already exists: {os.path.basename(f)}')
    print(f'\nCopied {copied} files')
    print('Files in working dir:')
    for f in sorted(os.listdir(WORK_DIR)):
        if not f.startswith('.'):
            print(f'  {f}')

In [ ]:
# CELL 3: Verify new traci_env.py is loaded
import os
src = open('/kaggle/working/traci_env.py').read()
if '_safe_close' in src:
    print('OK - traci_env.py has _safe_close fix built in')
    print('Ready to train')
else:
    print('WARNING - old traci_env.py detected, re-upload the latest version')

In [ ]:
# CELL 4: Install Python packages
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', 'numpy', '-q'])
print('Python packages ready')

In [ ]:
# CELL 5: Quick environment test (50 steps only, not real training)
import os, sys, numpy as np
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/kaggle/working')
sys.path.append('/usr/share/sumo/tools')
from gcn_encoder import REAL_JUNCTIONS, RAW_OBS_DIM, ACT_DIM
from traci_env import SUMOEnv, SUMO_CFG, _SCRIPTS_DIR
print(f'SUMO files dir : {_SCRIPTS_DIR}')
print(f'run.sumocfg    : {SUMO_CFG}')
print(f'Config exists  : {os.path.exists(SUMO_CFG)}')
print()
print('Running 50-step environment test...')
env = SUMOEnv(warmup=50, ep_duration=100)
obs = env.reset()
rewards = []
for _ in range(50):
    acts = {jid: np.random.randint(0, ACT_DIM[jid]) for jid in REAL_JUNCTIONS}
    obs, rews, done, info = env.step(acts)
    rewards.extend(rews.values())
    if done:
        break
env.close()
print(f'Reward range: min={min(rewards):.3f}  max={max(rewards):.3f}  mean={np.mean(rewards):.3f}')
print('Environment test PASSED' if max(rewards) <= 0 else 'WARNING: check reward range')

In [ ]:
# CELL 6: Stage 0 - Baseline training on final_routes.rou.xml
# Set EPISODES=5 to test first, then change to 300 for real training
import os, sys, torch
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/kaggle/working')
sys.path.append('/usr/share/sumo/tools')
from traci_env import train
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
EPISODES = 5
print(f'Device: {DEVICE}  |  Episodes: {EPISODES}')
ctrl = train(
    n_episodes = EPISODES,
    device     = DEVICE,
    save_dir   = '/kaggle/working/ckpt_base',
    use_gui    = False,
    curriculum = False,
)

In [ ]:
# CELL 7: Stage 1 - Off-peak training (low density, resumes from Stage 0)
# Set EPISODES=5 to test first, then change to 200 for real training
import os, sys, torch
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/kaggle/working')
sys.path.append('/usr/share/sumo/tools')
from traci_env import train
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
EPISODES = 5
print(f'Device: {DEVICE}  |  Episodes: {EPISODES}')
ctrl = train(
    n_episodes  = EPISODES,
    device      = DEVICE,
    save_dir    = '/kaggle/working/ckpt_low',
    resume_from = '/kaggle/working/ckpt_base',
    use_gui     = False,
    curriculum  = False,
)

In [ ]:
# CELL 8: Stage 2 - Moderate traffic training (resumes from Stage 1)
# Set EPISODES=5 to test first, then change to 200 for real training
import os, sys, torch
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/kaggle/working')
sys.path.append('/usr/share/sumo/tools')
from traci_env import train
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
EPISODES = 5
print(f'Device: {DEVICE}  |  Episodes: {EPISODES}')
ctrl = train(
    n_episodes  = EPISODES,
    device      = DEVICE,
    save_dir    = '/kaggle/working/ckpt_med',
    resume_from = '/kaggle/working/ckpt_low',
    use_gui     = False,
    curriculum  = False,
)

In [ ]:
# CELL 9: Stage 3 - Peak hour training (resumes from Stage 2)
# Set EPISODES=5 to test first, then change to 400 for real training
import os, sys, torch
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/kaggle/working')
sys.path.append('/usr/share/sumo/tools')
from traci_env import train
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
EPISODES = 5
print(f'Device: {DEVICE}  |  Episodes: {EPISODES}')
ctrl = train(
    n_episodes  = EPISODES,
    device      = DEVICE,
    save_dir    = '/kaggle/working/ckpt_high',
    resume_from = '/kaggle/working/ckpt_med',
    use_gui     = False,
    curriculum  = False,
)

In [ ]:
# CELL 10: Final evaluation across all 3 densities with same model
import os, sys, torch
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/kaggle/working')
sys.path.append('/usr/share/sumo/tools')
from traci_env import _evaluate
from mappo_atsc import MAPPOController
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
CKPT_DIR = '/kaggle/working/ckpt_high_best'
if not os.path.exists(CKPT_DIR):
    CKPT_DIR = '/kaggle/working/ckpt_high'
print(f'Loading from: {CKPT_DIR}')
ctrl = MAPPOController(device=DEVICE)
ctrl.load(CKPT_DIR)
print('\n=== Evaluation Results (same model on all densities) ===')
for density, label in [('low', 'Off-peak'), (None, 'Moderate (default)'), ('high', 'Peak hour')]:
    print(f'\nDensity: {label}')
    _evaluate(ctrl, n_reps=3, use_gui=False, density=density)

In [ ]:
# CELL 11: Plot training curves
import pandas as pd
import matplotlib.pyplot as plt
import os

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
stages = [
    ('ckpt_base', 'Stage 0: Baseline'),
    ('ckpt_low',  'Stage 1: Off-peak'),
    ('ckpt_med',  'Stage 2: Moderate'),
    ('ckpt_high', 'Stage 3: Peak hour'),
]
for ax, (ckpt, title) in zip(axes.flat, stages):
    log_path = f'/kaggle/working/{ckpt}/training_log.csv'
    if os.path.exists(log_path):
        df = pd.read_csv(log_path)
        ax.plot(df['episode'], df['avg_reward'], label='Reward', color='blue')
        ax2 = ax.twinx()
        ax2.plot(df['episode'], df['avg_travel_time'], label='Travel time', color='orange')
        ax.set_title(title)
        ax.set_xlabel('Episode')
        ax.set_ylabel('Reward', color='blue')
        ax2.set_ylabel('Travel time (s)', color='orange')
    else:
        ax.text(0.5, 0.5, f'No data\n{ckpt}', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves_all_stages.png', dpi=150)
plt.show()
print('Plot saved: training_curves_all_stages.png')